# Сравнительный анализ современных архитектур сегментации изображений

Домашнее задание №2, Попина Руслан, М25-555

## Цель работы

Разобраться, как устроены современные модели сегментации, чем они отличаются по способу получения масок, и научиться обоснованно выбирать архитектуру под конкретную задачу.

Рассматриваю SegFormer, Mask2Former, YOLACT / YOLO-Seg, SAM 2, Grounded SAM и Cutie. Опирался в основном на оригинальные статьи, ссылки в конце.

## 1. Иерархический Transformer в SegFormer

Обычный ViT режет картинку на патчи 16x16 и дальше во всех слоях работает с одной и той же сеткой токенов. На выходе одна карта признаков с разрешением 1/16. Для классификации этого хватает, там нужен один вектор на картинку. Для сегментации нужен ответ в каждом пикселе, и тут начинаются проблемы:

- мелкие детали теряются уже на входе: объект меньше патча становится частью одного токена, граница квантуется с шагом 16 пикселей;
- поднять разрешение, уменьшив патч, не выйдет: глобальное внимание стоит $O(N^2)$ по числу токенов. При патче 4x4 токенов в 16 раз больше, а внимание дороже в 256 раз;
- позиционные эмбеддинги ViT рассчитаны на фиксированный размер входа. В сегментации тестовые картинки часто другого размера, чем обучающие кропы (в Cityscapes кадры 1024x2048), эмбеддинги приходится интерполировать, и качество падает.

SegFormer решает это иерархическим энкодером Mix Transformer (MiT, версии B0-B5). По структуре он похож на CNN вроде ResNet, но внутри трансформерные блоки.

### Как получаются многоуровневые карты признаков

Энкодер состоит из 4 стадий. Каждая начинается с overlapped patch merging, по сути это свёртка с перекрытием: на первой стадии ядро 7 и шаг 4, дальше ядро 3 и шаг 2. Перекрытие нужно, чтобы соседние токены делили пиксели и не было швов на стыках патчей. На каждой следующей стадии разрешение падает в 2 раза, а каналов становится больше. В итоге получаются карты на 1/4, 1/8, 1/16 и 1/32 от входа (у B1-B5 с 64, 128, 320 и 512 каналами).

Внутри стадий два отличия от ViT:

- Efficient self-attention. Перед вниманием ключи и значения сжимаются по пространству в $R$ раз ($R$ = 64, 16, 4, 1 по стадиям), запросы остаются полного размера. Сложность падает с $O(N^2)$ до $O(N^2/R)$. Только поэтому можно делать внимание уже на карте 1/4: каждый пиксель по-прежнему смотрит на всю картинку, но по более грубой сетке.
- Mix-FFN вместо позиционного кодирования: $x_{out} = \mathrm{MLP}(\mathrm{GELU}(\mathrm{Conv}_{3\times3}(\mathrm{MLP}(x_{in})))) + x_{in}$. Depthwise-свёртка 3x3 сама даёт сети информацию о положении (через zero padding), поэтому явные эмбеддинги не нужны и разрешение на тесте можно менять. В статье на Cityscapes при переходе с 768x768 на 1024x2048 модель с позиционным кодированием теряет 3.3 mIoU, а с Mix-FFN только 0.7.

Декодер совсем простой (All-MLP): все 4 карты приводятся линейным слоем к одному числу каналов, апсемплятся до 1/4, конкатенируются, ещё один MLP их смешивает, и последний слой предсказывает класс каждого пикселя на сетке 1/4. До размера картинки маска доводится билинейной интерполяцией.

### Зачем нужны признаки разных масштабов

Разные уровни отвечают на разные вопросы. На 1/4 и 1/8 признаки локальные: видно, где именно проходит граница, сохраняются тонкие и мелкие объекты (столбы, знаки, пешеходы вдали). На 1/16 и 1/32 у каждого токена огромное рецептивное поле, там понятно, что это за сцена и к какому классу относится большая однородная область (дорога, небо, стена). Если брать только верхнюю карту, класс будет правильный, но границы рваные и мелочь пропадёт. Если только нижнюю, границы чёткие, но легко перепутать похожие по текстуре классы, например тротуар и дорогу, потому что не хватает контекста. К тому же объекты в одной сцене бывают очень разного размера, и одного масштаба на всех не хватает.

В статье есть анализ эффективного рецептивного поля: у MiT оно даже на ранних стадиях шире, чем у CNN (сравнивали с DeepLabv3+), а на 4-й стадии уже нелокальное. Поэтому декодеру не нужно ничего досчитывать (ASPP, большие свёртки), достаточно смешать уровни, где нижние дают локальное внимание, а верхние глобальное, и с этим справляются обычные MLP-слои.

### Плюсы и ограничения по сравнению с ViT

Плюсы:

- признаки с 1/4, поэтому точнее границы и мелкие объекты;
- вычисления растут с разрешением терпимо, можно подавать большие картинки;
- нет позиционных эмбеддингов фиксированного размера, разрешение легко менять;
- лёгкий декодер и линейка размеров от B0 до B5. SegFormer-B4 даёт 50.3 mIoU на ADE20K при 64M параметров, B5 даёт 84.0 mIoU на Cityscapes val.

Ограничения:

- внимание на ранних стадиях приближённое: при $R = 64$ ключи и значения сжаты в 8 раз по каждой стороне, тонкие детали в них смазываются, и локальные мелочи держатся в основном на свёртке в Mix-FFN;
- выход всё равно на 1/4, финальная маска получается интерполяцией, структуры толщиной в 1-2 пикселя теряются;
- энкодер не плоский, веса обычного ViT (MAE, CLIP, DINOv2) напрямую не взять, MiT предобучают отдельно на ImageNet. Плоский ViT проще масштабировать на огромных данных;
- в архитектуру заложено больше предположений (локальность, пирамида), при очень большом объёме данных плоский ViT может оказаться гибче;
- это всё ещё попиксельная классификация: экземпляры одного класса не разделяются, набор классов фиксирован;
- авторы сами пишут, что неясно, поместится ли даже самая маленькая модель на совсем слабые устройства.

## 2. Mask Queries в Mask2Former

Классическая семантическая сегментация (FCN, DeepLab, тот же SegFormer) - это классификация каждого пикселя. На выходе тензор $H \times W \times K$, softmax по $K$ классам и cross-entropy по пикселям. Число выходов жёстко равно числу классов, поэтому два человека, стоящие рядом, получают одну метку, и разделить их нельзя. Для instance и panoptic приходится брать другие архитектуры (Mask R-CNN и т.п.).

MaskFormer, а за ним Mask2Former переходят к классификации масок: модель предсказывает набор из $N$ пар (маска, класс). $N$ фиксировано (по умолчанию 100) и не связано с числом классов. Одна архитектура и один лосс подходят для semantic, instance и panoptic, меняются только разметка и постобработка.

### Зачем нужны Mask Queries

Mask query - обучаемый вектор размерности 256, по сути слот под один сегмент: объект («thing») или область («stuff»). В трансформерном декодере запросы через cross-attention собирают информацию из признаков картинки, а через self-attention общаются между собой, чтобы два запроса не описывали один и тот же объект. В Mask2Former начальные признаки запросов тоже обучаемые и сразу получают лосс, то есть ещё до декодера работают как грубые предложения регионов, примерно как RPN.

### Как получаются маски и классы

Схема из трёх частей: backbone (ResNet или Swin), pixel decoder на multi-scale deformable attention и transformer decoder. Pixel decoder выдаёт пирамиду признаков (1/32, 1/16, 1/8) для декодера и попиксельные эмбеддинги $\varepsilon_{pixel}$ на 1/4.

У каждого выходного запроса две головы:

- линейный слой даёт вероятности $K+1$ классов, где лишний класс - «нет объекта»;
- MLP даёт mask embedding $\varepsilon_{mask}$.

Маска $i$-го запроса - скалярное произведение его mask embedding с эмбеддингом каждого пикселя:

$$m_i[h,w] = \sigma\left(\varepsilon_{mask,i}^\top \, \varepsilon_{pixel}[h,w]\right)$$

Запрос здесь работает как динамический фильтр 1x1 по карте пикселей. Маски бинарные (sigmoid), а не softmax по классам, поэтому маски разных запросов не конкурируют за пиксель на этапе предсказания.

При обучении предсказаний $N$, а объектов в разметке меньше. Поэтому сначала венгерским алгоритмом каждому объекту подбирается своё предсказание по стоимости из вероятности класса и ошибки маски (BCE + dice). Сопоставленные запросы учатся на свой объект, остальные на класс «нет объекта». Ошибку маски считают не по всей карте, а по 12544 точкам (112x112), выбранным с упором на неуверенные области. Это уменьшает память на обучении в 3 раза (с 18 до 6 ГБ на картинку).

На инференсе для semantic берётся $\arg\max_c \sum_i p_i(c)\, m_i[h,w]$, для instance и panoptic каждый уверенный запрос даёт отдельный сегмент со своим классом.

### Masked Attention

В декодере DETR каждый запрос через cross-attention смотрит на всю картинку. Это медленно учится: запросу нужно много эпох, чтобы научиться концентрироваться на своём объекте, а внимание размазывается по фону и похожим объектам.

Mask2Former ограничивает cross-attention областью маски, которую этот же запрос предсказал на предыдущем слое:

$$X_l = \mathrm{softmax}\left(\mathcal{M}_{l-1} + Q_l K_l^\top\right) V_l + X_{l-1}$$

$$\mathcal{M}_{l-1}(x,y) = \begin{cases} 0, & M_{l-1}(x,y) = 1 \\ -\infty, & \text{иначе} \end{cases}$$

где $M_{l-1}$ - маска с предыдущего слоя, приведённая к разрешению $K_l$ и бинаризованная по порогу 0.5. Вне маски вес внимания нулевой.

Почему это уточняет маски:

- после каждого слоя декодера предсказывается промежуточная маска, и на неё тоже идёт лосс. Следующий слой смотрит только внутрь неё и собирает признаки именно объекта, без примеси фона. Обновлённый запрос даёт более точную маску, она снова ограничивает внимание, и так слой за слоем, от грубого к точному;
- авторы исходят из того, что для обновления запроса хватает локальных признаков, а контекст сцены запрос получает через self-attention с другими запросами;
- масштабы подаются по кругу: 9 слоёв: 3 круга по 3 масштаба (1/32, 1/16, 1/8). Каждый проход по пирамиде заканчивается на более детальной карте, так что маска уточняется и по разрешению;
- порядок внутри слоя поменян: сначала masked cross-attention, потом self-attention, чтобы запрос с первого же слоя брал информацию с картинки.

В коде есть деталь: если маска запроса пустая, ограничение для него снимают, иначе softmax по одним $-\infty$ дал бы NaN.

С одной архитектурой Mask2Former получил 57.8 PQ на COCO panoptic, 50.1 AP на COCO instance и 57.7 mIoU на ADE20K. Из ограничений авторы называют слабость на мелких объектах и то, что под каждую задачу модель всё равно обучают отдельно.

## 3. Прототипные маски в YOLACT и YOLO-Seg

Идея YOLACT - разделить экземплярную сегментацию на две параллельные задачи. Одна ветка строит небольшой набор масок, общих для всей картинки. Другая для каждого найденного объекта предсказывает, в какой пропорции эти общие маски смешать.

### Что такое прототипы

Protonet - небольшая полносвёрточная сеть на самом детальном уровне FPN (P3), выход апсемплится до 1/4 от входа. Для входа 550x550 это 138x138xk, где k = 32. Каждый канал - один прототип. Прототипы не привязаны к конкретному объекту и отдельно не размечаются: лосс считается только по итоговым маскам, а прототипы получаются сами. На последнем слое ReLU, сверху значения не ограничены, сильная активация означает уверенную область.

В статье видно, что сеть сама раскладывает картинку на базовые узоры: одни прототипы активируются на объектах по одну сторону от неявно выученной границы, другие на краях, третьи на фоне или на всём переднем плане. То, что полносвёрточная сеть различает положение, авторы объясняют zero padding'ом.

### Как из прототипов получается маска объекта

Детекционная голова для каждого анкора предсказывает класс, бокс и ещё k коэффициентов маски с tanh. tanh нужен, чтобы коэффициенты могли быть отрицательными: тогда прототип можно вычесть, например убрать соседний объект или фон. Грубо говоря, если есть прототип «передний план» и прототип «левая половина», то для объекта справа берём первый с плюсом, второй с минусом.

После NMS остаётся $n$ детекций с матрицей коэффициентов $C$ размера $n \times k$. Маски всех объектов собираются одним умножением:

$$M = \sigma\left(P C^\top\right), \quad P \in \mathbb{R}^{h \times w \times k}, \; C \in \mathbb{R}^{n \times k}$$

Маска объекта - это линейная комбинация прототипов, пропущенная через sigmoid. Потом маска обрезается по предсказанному боксу и бинаризуется. При обучении обрезают по GT-боксу, а лосс маски делят на площадь бокса, чтобы мелкие объекты не терялись на фоне крупных.

В YOLO-Seg от Ultralytics (YOLOv8-seg, YOLO11-seg) схема та же, это видно по коду головы `Segment`. Модуль `Proto` берёт карту P3 (шаг 8), делает свёртку, увеличивает в 2 раза транспонированной свёрткой и выдаёт 32 прототипа на 1/4 входа (160x160 для 640x640). В голове на каждом уровне есть отдельная ветка, которая для каждой позиции выдаёт 32 коэффициента, уже без tanh. Дальше `masks = coef @ protos`, порог по логиту 0 (то же, что sigmoid > 0.5) и обрезка по боксу. Отличается в основном детектор (anchor-free голова, другой backbone), а сборка масок такая же.

### Почему это быстрее двухэтапных моделей

Mask R-CNN работает последовательно: RPN предлагает регионы, для каждого RoIAlign заново вырезает и интерполирует признаки (14x14), затем маленькая FCN-голова отдельно для каждого региона выдаёт маску 28x28, которая растягивается на бокс. Маску нельзя начать считать, пока нет боксов, а стоимость растёт с числом регионов. Повторный пулинг признаков на каждый RoI как раз и тормозит.

В YOLACT:

- прототипы считаются один раз на картинку и параллельно с детекцией, они не ждут боксов;
- на объект приходится всего k чисел, их выдаёт та же голова, что класс и бокс, почти бесплатно;
- сборка масок всех объектов - одно матричное умножение и sigmoid, на GPU это очень быстро. Вся маскная ветка в статье занимает около 5 мс;
- никакого пулинга признаков на каждый регион;
- плюс Fast NMS, который считается матрично и на 12 мс быстрее обычного NMS почти без потери точности.

В итоге YOLACT даёт 29.8 mAP на COCO при 33.5 FPS на одной Titan Xp, а Mask R-CNN в своей статье работает около 5 FPS (зато точнее). Есть и бонус по качеству: маска строится в разрешении 138x138 на всю картинку, а не 28x28 на бокс, поэтому у крупных объектов границы аккуратнее.

Минусы следуют из той же схемы. Маска не выходит за бокс, поэтому плохой бокс сразу портит маску. Когда сильно перекрываются объекты одного класса, сети трудно развести их по прототипам (в статье это localization failure), а при слишком большом боксе в маску протекает соседний объект (leakage). 32 прототипа на 1/4 разрешения тоже ограничивают детализацию сложных форм.

## 4. Универсальная сегментация с помощью SAM 2

Модели из пунктов 1-3 обучаются на фиксированном наборе классов (19 в Cityscapes, 150 в ADE20K, 80 в COCO). Классификационная голова жёстко привязана к этому списку, и объект не из списка попадёт либо в чужой класс, либо в фон. У SAM и SAM 2 классов нет вообще, выход class-agnostic. Модель возвращает маску той области, на которую указала подсказка, а что именно сегментировать, решается уже на инференсе.

### Promptable segmentation

Подсказка - это точки (положительные и отрицательные), бокс или грубая маска. Модель должна вернуть корректную маску на любую подсказку, даже неоднозначную. Например, клик по пуговице может означать пуговицу, рубашку или человека целиком. Поэтому на неоднозначную подсказку (один клик) модель выдаёт несколько масок (часть, объект, целое) и для каждой оценку IoU, по которой выбирается лучшая. Дальше маску можно уточнять дополнительными кликами.

SAM 2 переносит это на видео: подсказка на одном кадре даёт маску объекта на всех кадрах, а ошибку можно поправить кликом на любом кадре. Картинка для SAM 2 - просто видео из одного кадра. Текстовых подсказок в SAM 2 нет.

Обобщает модель в основном за счёт данных: SA-1B (11 млн картинок, 1.1 млрд масок) и SA-V (50.9 тыс. видео, 642.6 тыс. masklet'ов). Оба датасета собраны через data engine, где модель помогает разметчикам, а новая разметка идёт на дообучение модели.

### Image Encoder, Prompt Encoder и Mask Decoder

Image encoder - тяжёлая часть. В SAM 2 это иерархический Hiera, предобученный через MAE (в первом SAM был обычный ViT). Он прогоняется один раз на кадр и выдаёт эмбеддинг, не зависящий от подсказки. Признаки с шагом 16 и 32 объединяются через FPN и идут дальше, а признаки с шагом 4 и 8 через skip connections попадают сразу в апсемплинг декодера, это помогает с мелкими деталями. Деление на тяжёлый энкодер и лёгкое всё остальное важно для интерактива: картинку кодируем один раз, а каждая подсказка обрабатывается быстро. Уже у SAM 1 prompt encoder и декодер работали около 50 мс в браузере на CPU.

Prompt encoder переводит подсказки в токены. Точки и боксы - позиционное кодирование координат плюс обученный эмбеддинг типа (положительная точка, отрицательная, угол бокса). Маска кодируется свёртками и поэлементно складывается с эмбеддингом кадра.

Mask decoder - лёгкий двусторонний трансформер. Токены подсказки и выходные токены через cross-attention смотрят на эмбеддинг картинки, а картинка в ответ смотрит на токены. Потом эмбеддинг апсемплится, и маска получается скалярным произведением выходного токена с каждым пикселем (та же идея, что в Mask2Former). Отдельные головы предсказывают IoU каждой маски, а в SAM 2 ещё и то, есть ли объект на кадре вообще (если он закрыт или ушёл из кадра).

Для видео добавлена память. Memory encoder сжимает предсказанную маску свёртками и складывает с эмбеддингом кадра. Memory bank хранит память нескольких последних кадров и кадров с подсказками (обе очереди FIFO), плюс object pointers - компактные векторы объекта из выходных токенов декодера. Memory attention перед декодером обуславливает признаки текущего кадра на эту память, поэтому модель знает, какой объект вести, без новых кликов. По статье на видео нужно в 3 раза меньше взаимодействий, чем у прошлых подходов, а на картинках SAM 2 точнее SAM и в 6 раз быстрее.

### Объекты, которых не было в специализированном датасете

Плюсы:

- не нужно собирать датасет и переобучать модель под новый класс, достаточно клика или бокса. Модель видела очень разнообразные данные и хорошо понимает, где кончается объект, поэтому на незнакомых объектах обычно даёт нормальную маску;
- хороший инструмент разметки: быстро нарезать маски для нового датасета, а потом обучить на них лёгкую специализированную модель;
- в видео одна подсказка даёт трекинг по всем кадрам, ошибку можно поправить кликом;
- маски с хорошими границами.

Ограничения:

- нет семантики. SAM 2 не скажет, что это за объект, и сам не найдёт все объекты нужного класса. Подсказку должен дать человек или другая модель (как в Grounded SAM). В автоматическом режиме по сетке точек он режет всё подряд без меток;
- неоднозначность гранулярности: часть или целое. Модель выбирает маску по предсказанному IoU, а задаче может быть нужно другое;
- сильный сдвиг домена: медицина (КТ и МРТ с низким контрастом), микроскопия, спутниковые снимки, прозрачные и камуфлированные объекты. Если границу на снимке видит только специалист (опухоль), SAM заметно уступает специализированным моделям, нужно дообучение;
- тяжёлый image encoder, на слабом железе в реальном времени дорого;
- в видео, по словам самих авторов, модель может потерять объект при смене плана, путает объекты в толпе похожих и плохо ведёт очень тонкие быстро движущиеся объекты. Несколько объектов модель ведёт независимо друг от друга, общие у них только признаки энкодера.

## 5. Open-Vocabulary Segmentation в Grounded SAM

Хочется по тексту, например «dog» или «red car», получить маски нужных объектов. Ни одна из двух моделей сама этого не умеет. SAM даёт хорошие маски, но не понимает текст и не знает классов. Grounding DINO понимает текст и находит объекты, но выдаёт только боксы. Учить одну модель, которая сразу по тексту выдаёт маску, тяжело: датасетов с масками и произвольными описаниями мало, а данных для детекции и grounding (бокс + фраза) намного больше. Поэтому в Grounded SAM просто соединили две готовые предобученные модели: детектор ищет объекты, SAM рисует их точную форму. Совместно их не дообучают, и компоненты можно менять: вместо SAM взять HQ-SAM, добавить RAM для автоматических тегов или Stable Diffusion для inpainting.

### Grounding DINO и SAM

Grounding DINO - open-set детектор. Картинку кодирует Swin, текст кодирует BERT. Feature enhancer смешивает модальности через cross-attention (картинка смотрит на текст и наоборот). Language-guided query selection выбирает 900 токенов картинки, наиболее похожих на текст, и делает их запросами декодера. Cross-modality decoder уточняет запросы, обращаясь и к картинке, и к тексту. На выходе для каждого запроса бокс и сходство с каждым токеном текста. Класс тут определяется близостью к словам запроса, фиксированного списка нет. Без обучения на COCO модель даёт 52.5 AP на COCO.

SAM в этой схеме нужен только для того, чтобы превратить каждый бокс в маску.

### Как текст превращается в маски

1. Текст, например `"dog . red car ."`, кодируется BERT.
2. Grounding DINO по картинке и тексту выдаёт 900 кандидатов: бокс и сходство с токенами.
3. Остаются боксы, у которых максимальное сходство выше box threshold. Метка бокса - фраза из токенов со сходством выше text threshold, например «red car». При необходимости NMS.
4. Картинка один раз проходит image encoder SAM.
5. Каждый бокс идёт в prompt encoder, затем в mask decoder и даёт маску.
6. На выходе маска, фраза и уверенность детектора.

В статье Grounded SAM (Grounding DINO-Base + SAM-Huge) даёт 48.7 mean AP на zero-shot бенчмарке SegInW.

### Как ошибки поиска объектов влияют на итог

Схема каскадная, и SAM рисует маску только по тому, что ему дал детектор. Его ошибки SAM никак не исправляет:

- Если детектор пропустил объект, бокса нет и маски тоже. Recall всей системы не выше recall Grounding DINO. Чаще всего пропадают мелкие, сильно перекрытые объекты и редкие слова.
- Детектор может найти не то: на «cat» взять собаку, на «red car» любую машину (атрибуты и отношения он понимает хуже, чем категории). SAM нарисует для такого бокса аккуратную маску, и ошибка выглядит уверенной. Оценка IoU у SAM при этом высокая: она показывает качество маски, о классе SAM ничего не знает. Про ложные срабатывания пишут и сами авторы Grounding DINO.
- Если бокс слишком узкий, маска обрезана. Если слишком широкий или задевает соседа, в маску попадает соседний объект или SAM выбирает не тот. Один бокс на группу («people») даёт одну маску на всех вместо отдельных людей.
- Дубликаты боксов дают дублированные маски, без NMS один объект посчитается дважды.
- Когда в запросе несколько объектов, маска может быть хорошей, а подпись взята от другой фразы.
- С порогами тоже не всё просто. Если снизить box threshold, recall вырастет, но появятся ложные боксы, которые SAM честно превратит в маски. Универсального значения нет, порог подбирается под данные.

Итого SAM отвечает за точность границ у найденных объектов, а какие объекты вообще найдены, целиком зависит от детектора. Что с этим можно сделать: подобрать пороги и NMS, давать SAM вдобавок к боксу точки, проверять вырезанную маской область через CLIP, а при разметке оставлять человека в цикле (это предлагают и авторы).

## 6. Использование памяти в Cutie

В semi-supervised VOS маска объекта (одного или нескольких) дана на первом кадре, а дальше его нужно сегментировать на всех следующих кадрах.

### Почему нельзя обрабатывать кадры независимо

- Объект задан только маской на первом кадре, класса у него нет. Модель, которая видит один кадр, просто не знает, что сегментировать. Даже если на каждом кадре запускать детектор или SAM, объекты всё равно надо сопоставлять между кадрами.
- Если в кадре два похожих объекта (две зебры, игроки одной команды), по одному кадру не понять, какой из них наш, без истории тут никак.
- Объект может пропасть за другим на несколько кадров и вернуться, и по одному кадру не узнать, что это тот же объект.
- Внешний вид меняется (поворот, деформация, свет), нужно накапливать разные виды объекта.
- Независимые маски дрожат от кадра к кадру.

Поэтому VOS-модели хранят память о прошлых кадрах и масках. Cutie развивает XMem: у XMem память только пиксельная, Cutie добавляет к ней объектную.

### Pixel Memory

Пиксельная память взята из XMem. Для кадров памяти хранятся ключи $k$ (низкоуровневые признаки внешнего вида, по ним ищется соответствие) и значения $v$ (признаки вместе с маской объекта), плюс sensory memory $h$ - скрытое состояние, которое рекуррентно обновляется каждый кадр. Текущий кадр кодируется в запрос $q$, для каждого его пикселя считается сходство со всеми пикселями памяти (affinity $A^{pix}$), и из значений собирается readout:

$$R_0 = f_{fuse}\left(A^{pix} v + h\right)$$

Каждый пиксель текущего кадра ищет похожие пиксели в прошлом и наследует их метку. Это сопоставление снизу вверх: точное по пространству, хорошо держит границы, но каждый пиксель решает за себя. Если на кадре есть похожий объект-дистрактор, его пиксели тоже сильно похожи на пиксели цели в памяти, и readout становится шумным.

### Object Memory и объектные запросы

Object memory $S \in \mathbb{R}^{N \times C}$ ($N = 16$) - компактное описание объекта целиком. Каждый вектор - это взвешенный масочный пулинг признаков памяти $U$:

$$S_q = \frac{\sum_i U(i)\, W_q(i)}{\sum_i W_q(i)}$$

Веса $W_q$ предсказывает сеть, причём первая половина масок пулит только пиксели объекта, а вторая только фон. Так память хранит и то, как выглядит объект, и то, что вокруг него, включая возможные дистракторы. На инференсе векторы усредняются потоково по всем кадрам памяти, поэтому стоимость не растёт с длиной видео. Если объект на кадре закрыт и веса пулинга нулевые, вектор не меняется, то есть во время окклюзии память не портится.

Объектные запросы $X \in \mathbb{R}^{N \times C}$ - обучаемые векторы, как в Mask2Former. Перед обработкой кадра к ним прибавляется объектная память: $X_0 = X + S$. Так запросы с самого начала знают, какой объект искать. Интересно, что в абляции сами статические запросы почти ничего не дают (67.2 против 67.3 J&F), основную работу делает объектная память.

### Как взаимодействуют пространственная и объектная память

Их связывает object transformer из $L = 3$ блоков. На входе пиксельный readout $R_0$ (из pixel memory) и запросы $X_0$ (из object memory). В каждом блоке:

1. Masked cross-attention: запросы читают пиксельные признаки. Половина запросов смотрит только на область объекта по текущей оценке маски, вторая половина только на фон. Маска считается из $R_{l-1}$ проекцией, sigmoid и порогом 0.5. Похоже на masked attention из Mask2Former, только с явным делением на объект и фон.
2. Self-attention и FFN между запросами: объектные и фоновые запросы обмениваются информацией.
3. Обратный cross-attention: пиксельные признаки читают запросы, объектная семантика записывается обратно в пиксели.
4. FFN по пикселям (свёртки). Прямого внимания между пикселями нет, чтобы не платить квадратично за высокое разрешение.

Информация идёт в обе стороны: пиксели обновляют объектное представление по текущему кадру, а запросы потом чистят пиксельный readout, потому что уже знают, где объект и где фон. Pixel memory отвечает за точную локализацию и детали, object memory за устойчивое понимание того, какой объект мы ведём. Финальные признаки $R_L$ идут в декодер маски. В абляции без масок во внимании J&F падает с 67.3 до 63.8, а с маской только для объекта (без фоновой половины) 66.7, так что деление на объект и фон реально помогает.

### Пример

Футбольный матч, ведём одного игрока. Мимо проходит игрок той же команды и на несколько кадров его закрывает.

Если память только пиксельная, то пиксели формы второго игрока почти неотличимы от пикселей нашего, affinity высокая, и маска перескакивает на второго игрока или растекается на обоих. В память при этом записываются кадры с неправильной маской, и ошибка закрепляется. Когда наш игрок выходит из-за второго, модель продолжает вести не того.

В Cutie объектная память накоплена по всей истории и не менялась, пока наш игрок был закрыт. Второй игрок до этого был фоном, поэтому его признаки попали в фоновые векторы памяти. В object transformer фоновые запросы забирают дистрактор себе, объектные собирают признаки только из области, похожей на цель, и через обратное внимание это поправляет пиксельный readout. В итоге маска остаётся на нашем игроке. Такие сцены с окклюзиями и дистракторами собраны в датасете MOSE, и там Cutie на 8.7 J&F лучше XMem при похожей скорости и на 4.2 J&F лучше DeAOT, работая в 3 раза быстрее.

Но и Cutie ошибается: в приложении статьи есть пример, где ломаются все модели, включая её, когда одна птица закрывает других похожих птиц.

## Сравнительная таблица

| Модель | Задача | Как получается маска | Преимущества | Ограничения | Где применять |
|---|---|---|---|---|---|
| SegFormer | семантическая | иерархический энкодер даёт карты 1/4-1/32, MLP-декодер смешивает их и классифицирует каждый пиксель | простой и быстрый, точнее ViT на границах и мелочи, легко меняет разрешение, есть маленькие версии | нет экземпляров, фиксированные классы, выход на 1/4 | карта сцены для автопилота и роботов, спутниковые и медицинские снимки с фиксированными классами |
| Mask2Former | семантическая, экземплярная, паноптическая | N запросов, у каждого класс и маска (скалярное произведение с пиксельными эмбеддингами), masked attention уточняет маску от слоя к слою | одна архитектура на все три задачи, высокая точность | тяжёлый для реального времени, слабее на мелких объектах, классы фиксированы, под каждую задачу своё обучение | паноптическая сегментация, офлайн-обработка, где точность важнее скорости |
| YOLACT / YOLO-Seg | экземплярная | линейная комбинация k общих прототипов с коэффициентами объекта, обрезка по боксу | реальное время, маска почти бесплатна поверх детектора, простой деплой | маска ограничена боксом, хуже на перекрытиях и сложных формах, классы фиксированы | видеоаналитика, роботы, производство, edge-устройства |
| SAM 2 | по подсказке, картинки и видео | подсказка кодируется в токены, двусторонний декодер выдаёт несколько масок с оценкой IoU, в видео помогает память | нет списка классов, хорошие границы, трекинг по одной подсказке | нет меток классов, нужна подсказка, хуже на далёких доменах, тяжёлый энкодер | интерактивная разметка, редактирование фото и видео, часть других пайплайнов |
| Grounded SAM | по тексту (open-vocabulary) | Grounding DINO по тексту находит боксы, SAM строит по ним маски | произвольный текст без дообучения, компоненты легко заменить | ошибки детектора не исправляются, две тяжёлые модели, плохо с атрибутами | авторазметка датасетов, поиск объектов по описанию, редактирование изображений |
| Cutie | объекты в видео по маске первого кадра | пиксельная память даёт readout, объектная память через object transformer чистит его от дистракторов | устойчив к окклюзиям и похожим объектам, память не растёт с длиной видео | нужна маска на первом кадре, нет классов, при сильных перекрытиях всё равно ошибается | трекинг в видео, ротоскопинг и видеомонтаж, разметка видео |

## Вывод

Все шесть моделей делают сегментацию, но постановки у них разные. Главное при выборе - кто говорит модели, что сегментировать: список классов на обучении, клик пользователя, текст или маска на первом кадре. Дальше уже смотрю на остальное:

- Тип выхода. Нужен класс каждого пикселя - хватит попиксельной классификации (SegFormer). Нужны отдельные экземпляры или panoptic - объектные запросы (Mask2Former) или прототипы (YOLO-Seg). Нужна маска конкретного объекта по подсказке - SAM 2. В видео с заданным объектом нужна память (Cutie, SAM 2).
- Закрытый или открытый набор классов. Если классы известны и есть разметка, специализированная модель будет точнее и быстрее. Если классы заранее неизвестны или часто меняются, нужны SAM 2 (с подсказками) или Grounded SAM (с текстом).
- Скорость и железо. Реальное время на слабом устройстве - YOLO-Seg или маленький SegFormer. Офлайн и максимум точности - Mask2Former. Grounded SAM - это две тяжёлые модели подряд, для реального времени не годится.
- Сколько есть разметки в домене. Разметки мало - SAM 2 как инструмент разметки, потом обучение лёгкой модели. Домен сильно отличается от обучающих данных (медицина, микроскопия) - foundation-модели без дообучения работают хуже.
- Требования к границам и мелким объектам. Нужны многомасштабные признаки и высокое разрешение маски, прототипы и выход на 1/4 тут ограничивают.
- Видео. Если важны согласованность и идентичность объектов между кадрами, покадровая обработка не подходит, нужна память. Объектная память особенно помогает при окклюзиях и похожих объектах.
- Человек в цикле. Если пользователь может кликнуть, promptable-модель решает задачу вообще без обучения.

Для себя я бы выбирал так: карта сцены для робота или автопилота - SegFormer; подсчёт объектов на конвейере в реальном времени - YOLO-Seg; качественная паноптическая разметка офлайн - Mask2Former; разметка нового датасета - SAM 2, а если классы заданы списком, то Grounded SAM с проверкой человеком; ведение объекта в видео по маске первого кадра - Cutie или SAM 2.

## Источники

1. Xie E. et al. SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers. NeurIPS 2021. https://arxiv.org/abs/2105.15203
2. Dosovitskiy A. et al. An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale. ICLR 2021. https://arxiv.org/abs/2010.11929
3. Cheng B., Schwing A., Kirillov A. Per-Pixel Classification is Not All You Need for Semantic Segmentation (MaskFormer). NeurIPS 2021. https://arxiv.org/abs/2107.06278
4. Cheng B. et al. Masked-attention Mask Transformer for Universal Image Segmentation (Mask2Former). CVPR 2022. https://arxiv.org/abs/2112.01527
5. Carion N. et al. End-to-End Object Detection with Transformers (DETR). ECCV 2020. https://arxiv.org/abs/2005.12872
6. Bolya D. et al. YOLACT: Real-time Instance Segmentation. ICCV 2019. https://arxiv.org/abs/1904.02689
7. He K. et al. Mask R-CNN. ICCV 2017. https://arxiv.org/abs/1703.06870
8. Ultralytics YOLO, код головы `Segment` и модуля `Proto`. https://github.com/ultralytics/ultralytics
9. Kirillov A. et al. Segment Anything. ICCV 2023. https://arxiv.org/abs/2304.02643
10. Ravi N. et al. SAM 2: Segment Anything in Images and Videos. 2024. https://arxiv.org/abs/2408.00714, код и release notes: https://github.com/facebookresearch/sam2
11. Ryali C. et al. Hiera: A Hierarchical Vision Transformer without the Bells-and-Whistles. ICML 2023. https://arxiv.org/abs/2306.00989
12. Liu S. et al. Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection. 2023. https://arxiv.org/abs/2303.05499
13. Ren T. et al. Grounded SAM: Assembling Open-World Models for Diverse Visual Tasks. 2024. https://arxiv.org/abs/2401.14159
14. Cheng H. K. et al. Putting the Object Back into Video Object Segmentation (Cutie). CVPR 2024. https://arxiv.org/abs/2310.12982
15. Cheng H. K., Schwing A. XMem: Long-Term Video Object Segmentation with an Atkinson-Shiffrin Memory Model. ECCV 2022. https://arxiv.org/abs/2207.07115